# E-Commerce Lakehouse: Bronze Products

## Ingest and validate source products

Read the original products CSV, preserve source values as strings,
and add ingestion metadata. Save a managed Bronze Delta table
and compare its row count with the source.

## Ingest and refresh Bronze products

Read the complete products CSV, preserving source values as strings.
Add ingestion metadata and verify the expected 32,951 source rows
before replacing the Bronze table. Validate the saved row count.

In [ ]:
from pyspark.sql.functions import current_timestamp

raw_products_df = (
    spark.read
    .option("header", True)
    .csv("/Volumes/workspace/ecommerce/raw_files/olist_products_dataset.csv")
)

source_count = raw_products_df.count()
print("Source product rows:", source_count)

if source_count != 32951:
    raise ValueError(
        f"Expected 32,951 products; found {source_count}. "
        "Stopped before replacing the table."
    )

bronze_products_df = raw_products_df.withColumn(
    "ingestion_timestamp",
    current_timestamp()
)

table_name = "workspace.ecommerce.bronze_products"

(
    bronze_products_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(table_name)
)

saved_bronze_products_df = spark.table(table_name)
saved_count = saved_bronze_products_df.count()

if saved_count != source_count:
    raise ValueError("Saved Bronze product count does not match the source.")

print("Bronze products refreshed:", saved_count)

In [ ]:
print("Source product rows:", raw_products_df.count())
raw_products_df.printSchema()